# CSE598/EEE589 Project 05 — single-GPU baseline

This notebook is independent of notebook 01. It detects the assigned CUDA device, selects BF16 or a separately labeled FP16, names the PyTorch SDPA backend that actually runs, and times forward and forward-plus-backward separately. It does not call the CPU schedule and it does not time Ring, DistFlashAttn, Ulysses, or USP. Without CUDA it records a skip.

## Fresh Colab runtime

1. Open `notebooks/02_colab_single_gpu_baseline.ipynb` from the same GitHub repository in a new Colab runtime.
2. **Runtime → Change runtime type → GPU → Save.**
3. The configuration cell already has the repository URL and commit. Do not change it.
4. **Runtime → Run all.**
5. Download the baseline ZIP from the last cell. If the manifest says `skipped_no_cuda`, no throughput was measured.

The bounded lengths are 1024, 2048, 4096, 8192, and 16384. A longer length is a separate decision, not the default. FP16 results, if a T4 forces that dtype, stay labeled FP16.


## Configuration


In [ ]:
# These values clone the project on a fresh Colab runtime.
# Leave ZIP_PATH and PROJECT_DIR empty.
REPO_URL = "https://github.com/nazndev/long-context-attention-benchmark.git"
REPO_COMMIT = "ee3c2a1045f48b9ad77ca61f3d480f8cb669e8d6"
ZIP_PATH = ""
PROJECT_DIR = ""


## Independent bootstrap


In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path

def discover(start):
    root = Path(start).resolve()
    pool = [root, *root.parents]
    if root.is_dir():
        for child in root.iterdir():
            if child.is_dir() and child.name not in {".venv", ".git"}:
                pool.append(child)
                for grandchild in child.iterdir():
                    if grandchild.is_dir() and grandchild.name not in {".venv", ".git"}:
                        pool.append(grandchild)
    for candidate in pool:
        if (candidate / "src" / "spbench" / "__init__.py").is_file():
            return candidate
    return None

if ZIP_PATH:
    dest = Path("/content/long-context-attention-benchmark" if os.path.isdir("/content") else "colab_unzipped")
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall(dest)
    project = discover(dest)
elif REPO_URL:
    if not REPO_COMMIT:
        raise ValueError("Set REPO_COMMIT to a full commit hash. Do not track a moving branch.")
    dest = Path("/content/long-context-attention-benchmark" if os.path.isdir("/content") else "colab_clone")
    if not (dest / ".git").exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(dest)])
    subprocess.check_call(["git", "checkout", REPO_COMMIT], cwd=dest)
    project = discover(dest)
else:
    project = Path(PROJECT_DIR).resolve() if PROJECT_DIR else discover(Path.cwd())

if project is None:
    raise FileNotFoundError(
        "No project source found. On a fresh Colab runtime, upload the project ZIP and set ZIP_PATH, "
        "or set REPO_URL and REPO_COMMIT."
    )
os.chdir(project)
src = project / "src"
for entry in (src, project):
    entry_text = str(entry)
    if entry_text not in sys.path:
        sys.path.insert(0, entry_text)
print("Project directory:", project)


In [ ]:
import importlib.util
import sys
bootstrap_path = project / "scripts" / "colab_bootstrap.py"
spec = importlib.util.spec_from_file_location("colab_bootstrap", bootstrap_path)
bootstrap = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bootstrap)
bootstrap.install_project(project)
src = project / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))
import spbench
print("spbench", spbench.__version__)


## Dtype and backend support


In [ ]:
import json
from spbench.baseline import detect_baseline_support
support = detect_baseline_support()
public = {key: value for key, value in support.items() if key != "torch_dtype"}
print(json.dumps(public, indent=2, default=str))


## Small forward and gradient check, only if CUDA is ready


In [ ]:
from spbench.baseline import correctness_against_reference
from spbench.config import load_config
config = load_config(project / "configs" / "benchmark.yaml")
rows = correctness_against_reference(support, config["correctness_lengths"], config)
for row in rows:
    print(row)


## Bounded single-GPU sweep in subprocesses


In [ ]:
from spbench.baseline import sweep_subprocess
manifest = sweep_subprocess(str(project / "configs" / "benchmark.yaml"), str(project / "results" / "raw" / "baseline"))
print(manifest["support"].get("status"), manifest["support"].get("reason"))
print("backend:", manifest["support"].get("backend"), "dtype:", manifest["support"].get("dtype"))
print("GPU:", manifest["support"].get("gpu_names"), manifest["support"].get("gpu_architectures"))
print("memory MiB total/free:", manifest["support"].get("gpu_memory_mib"), manifest["support"].get("gpu_free_mib"))
for item in manifest.get("capacity", []):
    print("capacity", item["phase"], item["capacity_class"], "last_success", item["last_success_length"], "first_oom", item["first_confirmed_oom_length"], "tested_upper_bound", item.get("tested_upper_bound"))


## Plots only from measured success rows


In [ ]:
import json
from pathlib import Path
import pandas as pd
from spbench.plots import plot_memory, plot_throughput
from spbench.schema import read_jsonl

frames = []
for path in sorted((project / "results" / "raw" / "baseline").glob("*.jsonl")):
    frames.extend(read_jsonl(path))
frame = pd.DataFrame(frames) if frames else pd.DataFrame()
if frame.empty or "origin" not in frame.columns:
    measured = frame.iloc[0:0]
else:
    measured = frame[(frame["origin"] == "measured") & (frame["status"] == "success")].copy()
if measured.empty:
    print("No measured baseline rows. No throughput or memory figure was written.")
else:
    figure_dir = project / "results" / "figures" / "measured"
    for phase, group in measured.groupby("phase"):
        plot_throughput(group, figure_dir / f"single_gpu_throughput_{phase}.png")
        plot_memory(group, figure_dir / f"single_gpu_memory_{phase}.png")
        print("Wrote measured figures for", phase, "from", len(group), "rows.")
(project / "results" / "processed" / "baseline_support.json").write_text(json.dumps(public, indent=2, default=str))


## Export


In [ ]:
import zipfile
from datetime import datetime, timezone
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
archive_path = project / "results" / f"baseline_evidence_{stamp}.zip"
root = project / "results"
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in (project / "results" / "raw" / "baseline").glob("*"):
        if path.is_file():
            archive.write(path, path.relative_to(project))
    support_path = project / "results" / "processed" / "baseline_support.json"
    if support_path.is_file():
        archive.write(support_path, support_path.relative_to(project))
    figure_dir = project / "results" / "figures" / "measured"
    if figure_dir.is_dir():
        for path in figure_dir.glob("single_gpu_*.png"):
            archive.write(path, path.relative_to(project))
print("Baseline archive:", archive_path)
